<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this over the others because the starter data already shows a large, well-populated
candidate pool with a measurable, tier-adjusted gap: a majority of pages clear a basic visibility
bar, and a large slice of those sit at decent search positions but capture noticeably fewer clicks
than other pages in the same position tier (numbers in Section 3). That's a big pool plus a sharp,
comparable definition of "underperforming" — the shape this lane is built for.

Lane 2 (general refresh scoring) was my other candidate, since `declining_with_demand` is also a
large bucket here, but it overlaps heavily with the CTR-gap pages and gives a vaguer "trending down"
target. CTR-vs-tier gives me a more precise, position-adjusted starting definition, and I can still
fold decline/refresh signals in later as a secondary reason code once the core scoring works.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MuhammadEhtisham776/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("clients:", df["client_id"].nunique())


rows, cols: (30000, 44)
clients: 32


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision this improves:** among pages that already rank somewhere reasonable, which ones should
a content editor open first because their click-through rate is worse than what other pages in the
same position tier typically get?

**Who acts, and what do they do:** a FlyRank content reviewer with a limited weekly review slot
works down a ranked queue and, for each flagged page, checks title, meta description, and intent
match — then rewrites, monitors, or clears it.

**Cost of a wrong call:** the two mistakes cost differently. A false positive (flagged, but nothing's
really wrong) costs an editor maybe 10-15 minutes of wasted review. A false negative (a genuinely
under-capturing, high-traffic page that never surfaces) is worse — it keeps quietly losing clicks
until something bigger forces attention elsewhere. So the queue should lean toward not missing
high-impression pages, even at the cost of a few extra false positives — I'll say that trade-off
out loud rather than optimizing for plain accuracy.

**Why data/ML, not a flat rule:** a flat cutoff like "ctr below 0.5%" ignores that expected CTR
depends heavily on position tier (Section 4 shows this). The same cutoff means something very
different at a top-3 position than at position 15 — a flat rule would just flag whichever tier has
naturally lower CTR, not whichever pages underperform their own peers. Comparing each page against
its tier, and combining that gap with volume, position, and content signals into one ranked score,
is the kind of "many signals, tangled" problem this internship's framing guidance says ML earns its
place on.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)]
tier_median_ctr = visible.groupby("position_tier")["ctr"].median().sort_values(ascending=False)
print("Median CTR by position tier, visible pages only:")
print(tier_median_ctr)

Median CTR by position tier, visible pages only:
position_tier
page_1      0.24
top_3       0.20
striking    0.17
page_3_5    0.09
deep        0.00
Name: ctr, dtype: float64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
n = len(df)
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1) Size of the candidate pool: pages with real visibility
visible = df[df["impressions_90d"] >= 500]
print(f"Visible pages (impressions_90d >= 500): {len(visible):,} of {n:,} ({len(visible)/n:.1%})")

# 2) Size of the CTR-opportunity slice within that pool, tier-adjusted (pos 1-20, ctr < 0.5%)
low_ctr_visible = visible[(visible["avg_position"] > 0) & (visible["avg_position"] <= 20) & (visible["ctr"] < 0.5)]
print(f"Low-CTR visible candidates: {len(low_ctr_visible):,} of {n:,} total "
      f"({len(low_ctr_visible)/n:.1%}), {len(low_ctr_visible)/len(visible):.1%} of the visible pool")

# 3) Where those candidates sit: mostly at page_1, i.e. good position but still under-capturing clicks
print("\nPosition-tier breakdown of the low-CTR candidates:")
print(low_ctr_visible["position_tier"].value_counts())

Visible pages (impressions_90d >= 500): 16,726 of 30,000 (55.8%)
Low-CTR visible candidates: 9,759 of 30,000 total (32.5%), 58.3% of the visible pool

Position-tier breakdown of the low-CTR candidates:
position_tier
page_1      5588
striking    3812
top_3        344
page_3_5      15
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** observed, tier-adjusted CTR gaps ("this page's CTR sits well below other
pages at the same position tier"); directional patterns across content types or intents; a
decision-support ranked queue with reason codes ("review these pages first, in this order,
because...").

**What I can't claim:** that rewriting a title or meta description *will* recover clicks — that
needs an actual before/after experiment, not this observational data. I also can't claim I've found
a Google ranking factor, or that a CTR gap *proves* a title/meta problem rather than, say, a SERP
feature stealing the click, a snippet FlyRank doesn't fully control, or plain low-volume noise. I'll
keep minimum-impression filters in place so low-volume noise doesn't get relabeled "opportunity."

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

banned = {"trend_direction", "trend_pct"}
candidate_signals = [
    "ctr", "avg_position", "position_tier", "impressions_90d", "clicks_90d",
    "word_count", "content_type", "main_intent", "engagement_rate", "scroll_rate",
]
assert banned.isdisjoint(candidate_signals), "leakage risk: a label-trap column snuck into the candidate signals"
print("Candidate signals for this lane (no label-trap columns):")
print(candidate_signals)


Candidate signals for this lane (no label-trap columns):
['ctr', 'avg_position', 'position_tier', 'impressions_90d', 'clicks_90d', 'word_count', 'content_type', 'main_intent', 'engagement_rate', 'scroll_rate']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.